# try19res_py12 — Colab 学習ノートブック (F-1)

ローカルにGPUが無いので、学習だけ Colab(GPU) で回すためのノートブック。

**手順**
1. ランタイム → ランタイムのタイプを変更 → **GPU** を選択
2. 下の「GPU確認」を実行
3. コード取得（**A: GitHub clone** か **B: Drive のフォルダ** のどちらか）
4. 依存インストール
5. **スモーク学習**で短時間動作確認 → 問題なければ **本学習**
6. チェックポイントを Drive に保存（再接続しても消えないように）

> ローカルの VSCode で編集 → GitHub へ push → ここで clone、が一番楽です。

## 0. GPU確認

In [1]:
!nvidia-smi


Sun Jun  7 13:01:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. コード取得（A か B のどちらか一方を実行）

### A) GitHub から clone
事前にローカルの変更を push しておくこと（`try19res_py12` を含むコミット）。

In [3]:
# === 方法A: GitHub clone ===
REPO_URL = "https://github.com/Waipy252/resnet-dqn.git"
BRANCH = "main"

%cd /content
![ -d resnet-dqn ] && rm -rf resnet-dqn
!git clone --branch $BRANCH --depth 1 $REPO_URL
%cd /content/resnet-dqn/try19res_py12
!pwd && ls


/content
Cloning into 'resnet-dqn'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 36 (delta 9), reused 21 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 4.62 MiB | 30.49 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/resnet-dqn/try19res_py12
/content/resnet-dqn/try19res_py12
algo.py		     Dockerfile  pyproject.toml     test.py
calc_performance.py  docs	 README.md	    uv.lock
config.py	     eval.py	 requirements.txt   visualize.py
data.py		     main.py	 run_simulation.py
docker-compose.yml   notebooks	 server.py


## 2. 依存インストール
Colab には CUDA 版 torch がプリインストール済みなので **torch は触らない**（壊さない）。
旧 `gym` も入れない（gymnasium に移行済み）。

In [5]:
!pip install -q \
    "stable-baselines3==2.8.0" \
    "sb3-contrib==2.8.0" \
    "gymnasium" "shimmy" \
    "yfinance" "pandas-datareader"
print('done')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.0/93.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 41.9 MB/s eta 0:00:00
done


In [6]:
# import 確認 + デバイス確認
import torch, stable_baselines3, sb3_contrib, gymnasium
print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())
print('sb3', stable_baselines3.__version__, 'contrib', sb3_contrib.__version__)
import config
print('ALGO =', config.ALGO, '| REWARD_TYPE =', config.REWARD_TYPE, '| WINDOW =', config.WINDOW_SIZE)


torch 2.11.0+cu128 cuda? True
sb3 2.8.0 contrib 2.8.0
ALGO = qrdqn | REWARD_TYPE = dsr | WINDOW = 130


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3. スモーク学習（短時間で動作確認）
短い期間・少ステップで、データ取得→env→GPU学習→predict までを確認する。

In [7]:
import torch
from stable_baselines3.common.vec_env import DummyVecEnv
import config
from data import generate_env_data
from main import NikkeiEnv, ResNetFeatures
from algo import build_model, get_algo_class

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device, '| algo =', get_algo_class().__name__)

# 短い期間でデータ取得（動作確認用）
df = generate_env_data('2015-01-01', '2024-01-01', ticker=config.TICKER)
print('data rows =', len(df))

env = DummyVecEnv([lambda: NikkeiEnv(df)])
model = build_model(env, device, features_extractor_class=ResNetFeatures, tensorboard_log=None)
model.learn(total_timesteps=3000, progress_bar=True)

obs = env.reset()
action, _ = model.predict(obs, deterministic=True)
print('smoke OK: device =', device, ' predicted action =', int(action))


/usr/local/lib/python3.12/dist-packages/pandas_datareader/compat/__init__.py:11: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  PANDAS_VERSION = LooseVersion(pd.__version__)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


device = cuda | algo = QRDQN


/content/resnet-dqn/try19res_py12/data.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  test_data = yf.download(ticker, start=start, end=end)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[*********************100%***********************]  1 of 1 completed/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

/content/resnet-dqn/try19res_py12/data.py:21: FutureWarning: YF.download() has changed argument auto_adjust de

data rows = 2200
Using cuda device


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


...

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

アクション[0:買,1:待,2:売]:0, ステップ:1996, 累積リワード:-11.9983, 資産:1361469, リターン:0.0, トレード回数:354, 
明日: 33458,株価:33458

smoke OK: device = cuda  predicted action = 2


/tmp/ipykernel_705/664006183.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('smoke OK: device =', device, ' predicted action =', int(action))


## 4. 本学習
`main.py` をそのまま実行（config の設定で学習）。`config.TOTAL_TIMESTEPS`（既定35万）まで回り、
1万ステップごとに `nikkei_cp_..._<steps>_steps.zip` を保存する。

GPU メモリ/速度を見たいときは別タブのセルで `!nvidia-smi` を実行。

In [8]:
!python main.py


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/pandas_datareader/compat/__init__.py:11: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  PANDAS_VERSION = LooseVersion(pd.__version__)
データをダウンロード中...
/content/resnet-dqn/try19res_py12/data.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  test_data = yf.download(ticker, start=start, end=end)
[*********************100%***********************]  1 of 1 completed
/content/resnet-dqn/try19res_py12/data.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  us_10y = yf.download("^TNX

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 5. チェックポイントを Drive に保存
（方法A で clone した場合、保存先はランタイム上なので消える。Drive にコピーして永続化する。）

In [ ]:
import glob, os, shutil
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('drive mount skip:', e)

DST = '/content/drive/MyDrive/resnet-dqn-models'
os.makedirs(DST, exist_ok=True)
files = sorted(set(glob.glob('nikkei_cp_*.zip')))
for f in files:
    shutil.copy(f, DST)
print(f'copied {len(files)} files to {DST}')


## 6. （任意）評価
`eval.py` はテスト期間のデータを取得し、各チェックポイントをバックテストして指標を出す。
重いので動作確認だけなら飛ばしてよい。

In [ ]:
!python eval.py
